# EEGMAT clean EEG-only utility and privacy baselines

This notebook-style script:
- keeps only the 19 scalp EEG electrodes (no ECG and no A2-A1 reference channel),
- filters and resamples the continuous recordings,
- creates non-overlapping raw EEG windows,
- balances every subject-condition pair,
- saves raw windows for later privacy transformations,
- evaluates condition utility with subject-held-out cross-validation, and
- evaluates closed-set subject identification across recording conditions.

In [1]:
# Colab setup. Run this cell in Google Colab.
from google.colab import drive
drive.mount("/content/drive")

get_ipython().system(
    'pip -q install "mne>=1.8,<2" "scikit-learn>=1.4,<2"'
)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 31.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [7]:
import json
import re
from pathlib import Path

import mne
import numpy as np
import pandas as pd
from scipy.signal import welch
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

BASE_DIR = Path("/content/drive/MyDrive/URV_Datasets")
EEGMAT_DIR = BASE_DIR / "eegmat"
RAW_DIR = EEGMAT_DIR / "raw"
CACHE_DIR = EEGMAT_DIR / "clean_eeg_only_cache"
RESULTS_DIR = EEGMAT_DIR / "results"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RAW_WINDOWS_FILE = CACHE_DIR / "eegmat_clean_raw_windows_2s_250hz_1_80hz_balanced.npz"
METADATA_FILE = CACHE_DIR / "eegmat_clean_raw_windows_2s_250hz_1_80hz_balanced_metadata.csv"
FEATURE_FILE = CACHE_DIR / "eegmat_clean_features_2s_250hz_1_80hz_balanced.npz"
RESULTS_FILE = RESULTS_DIR / "eegmat_clean_eeg_only_1_80hz_baselines.csv"
FOLD_RESULTS_FILE = RESULTS_DIR / "eegmat_clean_eeg_only_1_80hz_utility_folds.csv"
SUMMARY_FILE = RESULTS_DIR / "eegmat_clean_eeg_only_1_80hz_summary.json"

SCALP_CHANNELS = [
    "EEG Fp1", "EEG Fp2", "EEG F3", "EEG F4", "EEG F7", "EEG F8",
    "EEG T3", "EEG T4", "EEG C3", "EEG C4", "EEG T5", "EEG T6",
    "EEG P3", "EEG P4", "EEG O1", "EEG O2", "EEG Fz", "EEG Cz",
    "EEG Pz",
]

WINDOW_SEC = 2.0
STEP_SEC = 2.0
L_FREQ = 1.0
H_FREQ = 80.0
TARGET_SFREQ = 250.0

print("Cache directory:", CACHE_DIR)
print("Results directory:", RESULTS_DIR)

Cache directory: /content/drive/MyDrive/URV_Datasets/eegmat/clean_eeg_only_cache
Results directory: /content/drive/MyDrive/URV_Datasets/eegmat/results


## 1. Discover and validate the EEGMAT recordings

In [8]:
def parse_eegmat_path(path):
    match = re.fullmatch(r"Subject(\d+)_(\d)\.edf", path.name)
    if match is None:
        return None

    subject_num = int(match.group(1))
    condition_code = int(match.group(2))
    if condition_code not in (1, 2):
        return None

    return {
        "path": str(path),
        "subject_id": f"Subject{subject_num:02d}",
        "subject_num": subject_num,
        "condition": condition_code - 1,
        "condition_name": "rest" if condition_code == 1 else "arithmetic",
    }


records = []
for path in sorted(RAW_DIR.rglob("*.edf")):
    record = parse_eegmat_path(path)
    if record is not None:
        records.append(record)

files_df = pd.DataFrame(records).sort_values(
    ["subject_num", "condition"]
).reset_index(drop=True)

assert len(files_df) == 72, f"Expected 72 EDF files, found {len(files_df)}"
assert files_df["subject_id"].nunique() == 36
assert files_df.groupby("subject_id")["condition"].nunique().eq(2).all()

print(files_df.head())
print("\nFiles per condition:")
print(files_df["condition_name"].value_counts())

                                                path subject_id  subject_num  \
0  /content/drive/MyDrive/URV_Datasets/eegmat/raw...  Subject00            0   
1  /content/drive/MyDrive/URV_Datasets/eegmat/raw...  Subject00            0   
2  /content/drive/MyDrive/URV_Datasets/eegmat/raw...  Subject01            1   
3  /content/drive/MyDrive/URV_Datasets/eegmat/raw...  Subject01            1   
4  /content/drive/MyDrive/URV_Datasets/eegmat/raw...  Subject02            2   

   condition condition_name  
0          0           rest  
1          1     arithmetic  
2          0           rest  
3          1     arithmetic  
4          0           rest  

Files per condition:
condition_name
rest          36
arithmetic    36
Name: count, dtype: int64


In [9]:
# Confirm that all files contain the exact scalp montage we intend to use.
channel_audit = []
for row in files_df.itertuples(index=False):
    raw_header = mne.io.read_raw_edf(row.path, preload=False, verbose=False)
    missing = sorted(set(SCALP_CHANNELS) - set(raw_header.ch_names))
    extra = sorted(set(raw_header.ch_names) - set(SCALP_CHANNELS))
    channel_audit.append({
        "file": Path(row.path).name,
        "n_channels_in_file": len(raw_header.ch_names),
        "missing_scalp_channels": missing,
        "excluded_channels": extra,
        "sfreq": float(raw_header.info["sfreq"]),
    })

channel_audit_df = pd.DataFrame(channel_audit)
assert channel_audit_df["missing_scalp_channels"].map(len).eq(0).all()

print("Unique excluded-channel sets:")
print(channel_audit_df["excluded_channels"].astype(str).value_counts())
print("\nSampling frequencies:")
print(channel_audit_df["sfreq"].value_counts())

Unique excluded-channel sets:
excluded_channels
['ECG ECG', 'EEG A2-A1']    72
Name: count, dtype: int64

Sampling frequencies:
sfreq
500.0    72
Name: count, dtype: int64


## 2. Build transformation-ready, non-overlapping raw windows

Filtering is performed on each continuous recording before windowing. We then
select only the scalp channels and resample to 250 Hz. The cache stores raw
windows rather than only features, allowing every later privacy transformation
to be applied to the same signal passed to the utility and attacker branches.

In [10]:
def load_clean_windows(path):
    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)

    missing = sorted(set(SCALP_CHANNELS) - set(raw.ch_names))
    if missing:
        raise ValueError(f"Missing scalp channels in {path}: {missing}")

    # Explicit channel selection removes ECG ECG and EEG A2-A1.
    raw.pick(SCALP_CHANNELS)
    raw.reorder_channels(SCALP_CHANNELS)

    # Filter the continuous signal to avoid window-edge filtering artefacts.
    raw.filter(L_FREQ, H_FREQ, fir_design="firwin", verbose=False)
    raw.resample(TARGET_SFREQ, npad="auto", verbose=False)

    data = raw.get_data().astype(np.float32)
    sfreq = float(raw.info["sfreq"])
    win_size = int(round(WINDOW_SEC * sfreq))
    step_size = int(round(STEP_SEC * sfreq))

    windows = []
    starts = []
    for start in range(0, data.shape[1] - win_size + 1, step_size):
        windows.append(data[:, start:start + win_size])
        starts.append(start)

    return np.stack(windows), np.asarray(starts, dtype=np.int64), sfreq


def build_raw_window_cache(files_df):
    windows_all = []
    metadata_rows = []
    sfreq_seen = set()

    for file_index, row in files_df.iterrows():
        print(
            f"[{file_index + 1:02d}/{len(files_df)}] "
            f"{row['subject_id']} {row['condition_name']}"
        )
        windows, starts, sfreq = load_clean_windows(row["path"])
        sfreq_seen.add(sfreq)

        windows_all.append(windows)
        for local_index, start in enumerate(starts):
            metadata_rows.append({
                "subject_id": row["subject_id"],
                "subject_num": int(row["subject_num"]),
                "condition": int(row["condition"]),
                "condition_name": row["condition_name"],
                "source_file": row["path"],
                "window_index_in_file": int(local_index),
                "start_sample_250hz": int(start),
                "end_sample_250hz": int(start + windows.shape[-1]),
                "start_sec": float(start / sfreq),
                "end_sec": float((start + windows.shape[-1]) / sfreq),
            })

    if sfreq_seen != {TARGET_SFREQ}:
        raise ValueError(f"Unexpected post-resampling frequencies: {sfreq_seen}")

    X_raw_all = np.concatenate(windows_all, axis=0)
    metadata_all = pd.DataFrame(metadata_rows)
    assert len(X_raw_all) == len(metadata_all)

    # Every subject-condition pair contributes the same number of windows.
    group_sizes = metadata_all.groupby(["subject_id", "condition"]).size()
    windows_per_group = int(group_sizes.min())
    print("Minimum windows per subject-condition:", windows_per_group)

    balanced_indices = []
    for _, group in metadata_all.groupby(["subject_id", "condition"], sort=True):
        chosen = rng.choice(
            group.index.to_numpy(), size=windows_per_group, replace=False
        )
        balanced_indices.extend(sorted(chosen.tolist()))

    balanced_indices = np.asarray(balanced_indices, dtype=np.int64)
    X_raw = X_raw_all[balanced_indices]
    metadata = metadata_all.loc[balanced_indices].reset_index(drop=True)
    metadata.insert(0, "window_id", np.arange(len(metadata), dtype=np.int64))

    return X_raw, metadata


if RAW_WINDOWS_FILE.exists() and METADATA_FILE.exists():
    print("Loading existing clean raw-window cache...")
    cached = np.load(RAW_WINDOWS_FILE, allow_pickle=False)
    X_raw = cached["X_raw"]
    channel_names = cached["channel_names"].astype(str).tolist()
    sfreq = float(cached["sfreq"])
    metadata = pd.read_csv(METADATA_FILE)
else:
    print("Building clean raw-window cache...")
    X_raw, metadata = build_raw_window_cache(files_df)
    channel_names = SCALP_CHANNELS.copy()
    sfreq = TARGET_SFREQ

    np.savez_compressed(
        RAW_WINDOWS_FILE,
        X_raw=X_raw,
        channel_names=np.asarray(channel_names),
        sfreq=np.asarray(sfreq),
    )
    metadata.to_csv(METADATA_FILE, index=False)

assert X_raw.ndim == 3
assert X_raw.shape[1] == 19
assert channel_names == SCALP_CHANNELS
assert len(X_raw) == len(metadata)
assert metadata.groupby(["subject_id", "condition"]).size().nunique() == 1
assert not any("ECG" in name.upper() for name in channel_names)
assert "EEG A2-A1" not in channel_names

print("X_raw shape (windows, scalp channels, samples):", X_raw.shape)
print("Sampling frequency:", sfreq)
print("Channels:", channel_names)
print("Condition counts:")
print(metadata["condition_name"].value_counts())
print("Windows per subject-condition:",
      metadata.groupby(["subject_id", "condition"]).size().unique())

Building clean raw-window cache...
[01/72] Subject00 rest
[02/72] Subject00 arithmetic
[03/72] Subject01 rest
[04/72] Subject01 arithmetic
[05/72] Subject02 rest
[06/72] Subject02 arithmetic
[07/72] Subject03 rest
[08/72] Subject03 arithmetic
[09/72] Subject04 rest
[10/72] Subject04 arithmetic
[11/72] Subject05 rest
[12/72] Subject05 arithmetic
[13/72] Subject06 rest
[14/72] Subject06 arithmetic
[15/72] Subject07 rest
[16/72] Subject07 arithmetic
[17/72] Subject08 rest
[18/72] Subject08 arithmetic
[19/72] Subject09 rest
[20/72] Subject09 arithmetic
[21/72] Subject10 rest
[22/72] Subject10 arithmetic
[23/72] Subject11 rest
[24/72] Subject11 arithmetic
[25/72] Subject12 rest
[26/72] Subject12 arithmetic
[27/72] Subject13 rest
[28/72] Subject13 arithmetic
[29/72] Subject14 rest
[30/72] Subject14 arithmetic
[31/72] Subject15 rest
[32/72] Subject15 arithmetic
[33/72] Subject16 rest
[34/72] Subject16 arithmetic
[35/72] Subject17 rest
[36/72] Subject17 arithmetic
[37/72] Subject18 rest
[38/72

## 3. Extract identical EEG features for utility and privacy

In [11]:
BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "low_gamma": (30.0, 45.0),
    "high_gamma": (45.0, 80.0),
}


def extract_features(X_windows, sfreq, batch_size=256):
    """Absolute and relative bandpower plus time-domain features per channel."""
    output = []

    for batch_start in range(0, len(X_windows), batch_size):
        batch = X_windows[batch_start:batch_start + batch_size]
        freqs, psd = welch(
            batch,
            fs=sfreq,
            nperseg=min(256, batch.shape[-1]),
            axis=-1,
        )

        total_mask = (freqs >= 1.0) & (freqs <= 80.0)
        total_power = np.trapezoid(psd[..., total_mask], freqs[total_mask], axis=-1)
        total_power = np.maximum(total_power, np.finfo(np.float32).eps)

        feature_blocks = []
        for low, high in BANDS.values():
            mask = (freqs >= low) & (freqs < high)
            absolute = np.trapezoid(psd[..., mask], freqs[mask], axis=-1)
            relative = absolute / total_power
            feature_blocks.extend([np.log1p(absolute), relative])

        feature_blocks.extend([
            batch.mean(axis=-1),
            batch.std(axis=-1),
            np.sqrt(np.mean(np.square(batch), axis=-1)),
        ])

        batch_features = np.concatenate(feature_blocks, axis=1)
        output.append(batch_features.astype(np.float32))

    return np.concatenate(output, axis=0)


if FEATURE_FILE.exists():
    print("Loading cached clean features...")
    feature_cache = np.load(FEATURE_FILE, allow_pickle=False)
    X_features = feature_cache["X_features"]
else:
    print("Extracting clean EEG-only features...")
    X_features = extract_features(X_raw, sfreq)
    np.savez_compressed(FEATURE_FILE, X_features=X_features)

y_condition = metadata["condition"].to_numpy(dtype=np.int64)
subject_ids = metadata["subject_id"].to_numpy()

subject_encoder = LabelEncoder()
y_subject = subject_encoder.fit_transform(subject_ids)

assert np.isfinite(X_features).all()
assert len(X_features) == len(metadata)

print("Feature matrix shape:", X_features.shape)
print("Utility chance accuracy:", 1 / len(np.unique(y_condition)))
print("Privacy chance accuracy:", 1 / len(np.unique(y_subject)))

Extracting clean EEG-only features...
Feature matrix shape: (2232, 285)
Utility chance accuracy: 0.5
Privacy chance accuracy: 0.027777777777777776


## 4. Models and metrics

In [13]:
def make_models():
    return {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ]),
        "Random Forest": RandomForestClassifier(
            n_estimators=500,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "SVM RBF": Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", SVC(
                kernel="rbf",
                class_weight="balanced",
                C=1.0,
                gamma="scale",
            )),
        ]),
    }


def metric_row(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }

## 5. Utility baseline: subject-held-out 5-fold evaluation

Each test fold contains entirely unseen subjects. Every subject has equal rest
and arithmetic windows, so all folds contain both utility classes.

In [14]:
utility_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

utility_fold_rows = []
utility_splits = []

for fold, (train_idx, test_idx) in enumerate(
    utility_cv.split(X_features, y_condition, groups=subject_ids), start=1
):
    train_subjects = set(subject_ids[train_idx])
    test_subjects = set(subject_ids[test_idx])
    assert train_subjects.isdisjoint(test_subjects)

    # Save the exact folds for reuse in every privacy-transformation experiment.
    utility_splits.append({
        "fold": fold,
        "train_window_ids": train_idx.tolist(),
        "test_window_ids": test_idx.tolist(),
        "train_subjects": sorted(train_subjects),
        "test_subjects": sorted(test_subjects),
    })

    for model_name, model in make_models().items():
        fitted = clone(model)
        fitted.fit(X_features[train_idx], y_condition[train_idx])
        prediction = fitted.predict(X_features[test_idx])

        utility_fold_rows.append({
            "task": "utility",
            "protocol": "5-fold subject-held-out",
            "direction": "condition classification",
            "model": model_name,
            "fold": fold,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_train_subjects": len(train_subjects),
            "n_test_subjects": len(test_subjects),
            "chance_accuracy": 0.5,
            **metric_row(y_condition[test_idx], prediction),
        })

utility_folds_df = pd.DataFrame(utility_fold_rows)
utility_folds_df.to_csv(FOLD_RESULTS_FILE, index=False)

utility_summary = (
    utility_folds_df
    .groupby(["task", "protocol", "direction", "model", "chance_accuracy"], as_index=False)
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        weighted_f1_mean=("weighted_f1", "mean"),
        weighted_f1_std=("weighted_f1", "std"),
        n_evaluations=("fold", "count"),
    )
)

display(utility_folds_df)
display(utility_summary)

,task,protocol,direction,model,fold,n_train,n_test,n_train_subjects,n_test_subjects,chance_accuracy,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,utility,5-fold subject-held-out,condition classification,Logistic Regression,1,1736,496,28,8,0.5,0.643145,0.643145,0.635244,0.635244
1,utility,5-fold subject-held-out,condition classification,Random Forest,1,1736,496,28,8,0.5,0.679435,0.679435,0.673109,0.673109
2,utility,5-fold subject-held-out,condition classification,SVM RBF,1,1736,496,28,8,0.5,0.649194,0.649194,0.631047,0.631047
3,utility,5-fold subject-held-out,condition classification,Logistic Regression,2,1798,434,29,7,0.5,0.693548,0.693548,0.693351,0.693351
4,utility,5-fold subject-held-out,condition classification,Random Forest,2,1798,434,29,7,0.5,0.728111,0.728111,0.719509,0.719509
5,utility,5-fold subject-held-out,condition classification,SVM RBF,2,1798,434,29,7,0.5,0.711982,0.711982,0.709873,0.709873
6,utility,5-fold subject-held-out,condition classification,Logistic Regression,3,1798,434,29,7,0.5,0.654378,0.654378,0.653782,0.653782
7,utility,5-fold subject-held-out,condition classification,Random Forest,3,1798,434,29,7,0.5,0.762673,0.762673,0.761456,0.761456
8,utility,5-fold subject-held-out,condition classification,SVM RBF,3,1798,434,29,7,0.5,0.647465,0.647465,0.641711,0.641711
9,utility,5-fold subject-held-out,condition classification,Logistic Regression,4,1798,434,29,7,0.5,0.698157,0.698157,0.698078,0.698078


,task,protocol,direction,model,chance_accuracy,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,n_evaluations
0,utility,5-fold subject-held-out,condition classification,Logistic Regression,0.5,0.679320,0.028614,0.679320,0.028614,0.677551,0.031268,0.677551,0.031268,5
1,utility,5-fold subject-held-out,condition classification,Random Forest,0.5,0.716993,0.047289,0.716993,0.047289,0.712280,0.050164,0.712280,0.050164,5
2,utility,5-fold subject-held-out,condition classification,SVM RBF,0.5,0.681452,0.034365,0.681452,0.034365,0.674907,0.038788,0.674907,0.038788,5


## 6. Privacy baseline: cross-condition subject identification

This is a closed-set attack: all 36 identities are represented during attacker
training, but training and testing use different recording conditions/files.
We evaluate both directions and summarize their mean and variation.

In [15]:
privacy_rows = []
directions = [
    (0, 1, "train rest -> test arithmetic"),
    (1, 0, "train arithmetic -> test rest"),
]

for train_condition, test_condition, direction in directions:
    train_idx = np.flatnonzero(y_condition == train_condition)
    test_idx = np.flatnonzero(y_condition == test_condition)

    assert set(subject_ids[train_idx]) == set(subject_ids[test_idx])
    assert set(metadata.iloc[train_idx]["source_file"]).isdisjoint(
        set(metadata.iloc[test_idx]["source_file"])
    )

    for model_name, model in make_models().items():
        fitted = clone(model)
        fitted.fit(X_features[train_idx], y_subject[train_idx])
        prediction = fitted.predict(X_features[test_idx])

        scores = metric_row(y_subject[test_idx], prediction)
        privacy_rows.append({
            "task": "privacy",
            "protocol": "cross-condition closed-set subject ID",
            "direction": direction,
            "model": model_name,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_subjects": len(subject_encoder.classes_),
            "chance_accuracy": 1 / len(subject_encoder.classes_),
            **scores,
            "leakage_over_chance": scores["accuracy"] * len(subject_encoder.classes_),
        })

privacy_df = pd.DataFrame(privacy_rows)

privacy_summary = (
    privacy_df
    .groupby(["task", "protocol", "model", "chance_accuracy"], as_index=False)
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        weighted_f1_mean=("weighted_f1", "mean"),
        weighted_f1_std=("weighted_f1", "std"),
        leakage_over_chance_mean=("leakage_over_chance", "mean"),
        n_evaluations=("direction", "count"),
    )
)

display(privacy_df)
display(privacy_summary)

,task,protocol,direction,model,n_train,n_test,n_subjects,chance_accuracy,accuracy,balanced_accuracy,macro_f1,weighted_f1,leakage_over_chance
0,privacy,cross-condition closed-set subject ID,train rest -> test arithmetic,Logistic Regression,1116,1116,36,0.027778,0.541219,0.541219,0.528472,0.528472,19.483871
1,privacy,cross-condition closed-set subject ID,train rest -> test arithmetic,Random Forest,1116,1116,36,0.027778,0.344086,0.344086,0.316740,0.316740,12.387097
2,privacy,cross-condition closed-set subject ID,train rest -> test arithmetic,SVM RBF,1116,1116,36,0.027778,0.476703,0.476703,0.470205,0.470205,17.161290
3,privacy,cross-condition closed-set subject ID,train arithmetic -> test rest,Logistic Regression,1116,1116,36,0.027778,0.503584,0.503584,0.483114,0.483114,18.129032
4,privacy,cross-condition closed-set subject ID,train arithmetic -> test rest,Random Forest,1116,1116,36,0.027778,0.370072,0.370072,0.324900,0.324900,13.322581
5,privacy,cross-condition closed-set subject ID,train arithmetic -> test rest,SVM RBF,1116,1116,36,0.027778,0.427419,0.427419,0.417713,0.417713,15.387097


,task,protocol,model,chance_accuracy,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,leakage_over_chance_mean,n_evaluations
0,privacy,cross-condition closed-set subject ID,Logistic Regression,0.027778,0.522401,0.026612,0.522401,0.026612,0.505793,0.032073,0.505793,0.032073,18.806452,2
1,privacy,cross-condition closed-set subject ID,Random Forest,0.027778,0.357079,0.018375,0.357079,0.018375,0.320820,0.005770,0.320820,0.005770,12.854839,2
2,privacy,cross-condition closed-set subject ID,SVM RBF,0.027778,0.452061,0.034848,0.452061,0.034848,0.443959,0.037117,0.443959,0.037117,16.274194,2


## 7. Save clean baseline results and fixed evaluation design

In [16]:
utility_summary["result_level"] = "summary"
privacy_summary["result_level"] = "summary"

baseline_summary = pd.concat(
    [utility_summary, privacy_summary],
    ignore_index=True,
    sort=False,
)
baseline_summary.to_csv(RESULTS_FILE, index=False)

split_file = CACHE_DIR / "eegmat_clean_fixed_splits.json"
with open(split_file, "w") as handle:
    json.dump({
        "random_state": RANDOM_STATE,
        "utility_splits": utility_splits,
        "privacy_protocol": {
            "type": "cross-condition closed-set subject identification",
            "directions": [item[2] for item in directions],
        },
    }, handle, indent=2)

summary = {
    "dataset": "EEGMAT",
    "baseline_version": "clean_eeg_only_v1",
    "n_subjects": int(metadata["subject_id"].nunique()),
    "n_conditions": int(metadata["condition"].nunique()),
    "n_windows": int(len(X_raw)),
    "window_seconds": WINDOW_SEC,
    "step_seconds": STEP_SEC,
    "sampling_frequency_hz": sfreq,
    "filter_hz": [L_FREQ, H_FREQ],
    "n_channels": len(channel_names),
    "channels": channel_names,
    "explicitly_excluded_channels": ["EEG A2-A1", "ECG ECG"],
    "raw_window_shape": list(X_raw.shape),
    "feature_shape": list(X_features.shape),
    "windows_per_subject_condition": int(
        metadata.groupby(["subject_id", "condition"]).size().iloc[0]
    ),
    "utility_protocol": "5-fold subject-held-out",
    "privacy_protocol": "bidirectional cross-condition closed-set subject ID",
    "raw_windows_file": str(RAW_WINDOWS_FILE),
    "metadata_file": str(METADATA_FILE),
    "feature_file": str(FEATURE_FILE),
    "fixed_splits_file": str(split_file),
    "baseline_results_file": str(RESULTS_FILE),
    "utility_fold_results_file": str(FOLD_RESULTS_FILE),
}

with open(SUMMARY_FILE, "w") as handle:
    json.dump(summary, handle, indent=2)

print("Saved baseline summary:", RESULTS_FILE)
print("Saved utility fold results:", FOLD_RESULTS_FILE)
print("Saved fixed splits:", split_file)
print("Saved run metadata:", SUMMARY_FILE)
display(baseline_summary)

Saved baseline summary: /content/drive/MyDrive/URV_Datasets/eegmat/results/eegmat_clean_eeg_only_1_80hz_baselines.csv
Saved utility fold results: /content/drive/MyDrive/URV_Datasets/eegmat/results/eegmat_clean_eeg_only_1_80hz_utility_folds.csv
Saved fixed splits: /content/drive/MyDrive/URV_Datasets/eegmat/clean_eeg_only_cache/eegmat_clean_fixed_splits.json
Saved run metadata: /content/drive/MyDrive/URV_Datasets/eegmat/results/eegmat_clean_eeg_only_1_80hz_summary.json


,task,protocol,direction,model,chance_accuracy,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,n_evaluations,result_level,leakage_over_chance_mean
0,utility,5-fold subject-held-out,condition classification,Logistic Regression,0.500000,0.679320,0.028614,0.679320,0.028614,0.677551,0.031268,0.677551,0.031268,5,summary,NaN
1,utility,5-fold subject-held-out,condition classification,Random Forest,0.500000,0.716993,0.047289,0.716993,0.047289,0.712280,0.050164,0.712280,0.050164,5,summary,NaN
2,utility,5-fold subject-held-out,condition classification,SVM RBF,0.500000,0.681452,0.034365,0.681452,0.034365,0.674907,0.038788,0.674907,0.038788,5,summary,NaN
3,privacy,cross-condition closed-set subject ID,NaN,Logistic Regression,0.027778,0.522401,0.026612,0.522401,0.026612,0.505793,0.032073,0.505793,0.032073,2,summary,18.806452
4,privacy,cross-condition closed-set subject ID,NaN,Random Forest,0.027778,0.357079,0.018375,0.357079,0.018375,0.320820,0.005770,0.320820,0.005770,2,summary,12.854839
5,privacy,cross-condition closed-set subject ID,NaN,SVM RBF,0.027778,0.452061,0.034848,0.452061,0.034848,0.443959,0.037117,0.443959,0.037117,2,summary,16.274194


## Interpretation rule

EEGMAT is suitable for the privacy-preserving stage if:
1. subject-held-out utility remains meaningfully above 50% chance, and
2. cross-condition subject identification remains meaningfully above 1/36
   (2.78%) chance after ECG and A2-A1 removal.

Do not tune privacy transformations on the held-out utility folds. The saved
folds and cross-condition privacy directions should be reused unchanged for
every transformation strength.

In [17]:
import json

summary_path = (
    "/content/drive/MyDrive/URV_Datasets/eegmat/results/"
    "eegmat_clean_eeg_only_1_80hz_summary.json"
)

with open(summary_path, "r") as file:
    saved_summary = json.load(file)

saved_summary

{'dataset': 'EEGMAT',
 'baseline_version': 'clean_eeg_only_v1',
 'n_subjects': 36,
 'n_conditions': 2,
 'n_windows': 2232,
 'window_seconds': 2.0,
 'step_seconds': 2.0,
 'sampling_frequency_hz': 250.0,
 'filter_hz': [1.0, 80.0],
 'n_channels': 19,
 'channels': ['EEG Fp1',
  'EEG Fp2',
  'EEG F3',
  'EEG F4',
  'EEG F7',
  'EEG F8',
  'EEG T3',
  'EEG T4',
  'EEG C3',
  'EEG C4',
  'EEG T5',
  'EEG T6',
  'EEG P3',
  'EEG P4',
  'EEG O1',
  'EEG O2',
  'EEG Fz',
  'EEG Cz',
  'EEG Pz'],
 'explicitly_excluded_channels': ['EEG A2-A1', 'ECG ECG'],
 'raw_window_shape': [2232, 19, 500],
 'feature_shape': [2232, 285],
 'windows_per_subject_condition': 31,
 'utility_protocol': '5-fold subject-held-out',
 'privacy_protocol': 'bidirectional cross-condition closed-set subject ID',
 'raw_windows_file': '/content/drive/MyDrive/URV_Datasets/eegmat/clean_eeg_only_cache/eegmat_clean_raw_windows_2s_250hz_1_80hz_balanced.npz',
 'metadata_file': '/content/drive/MyDrive/URV_Datasets/eegmat/clean_eeg_only_